# PART 1: Text Processing and Exploratory Data Analysis 

## Task 1. Data preparation

1. As a first step, you must pre-process the documents. In particular, for the text fields (title,
description) you should:

    - Removing stop words
    - Tokenization
    - Removing punctuation marks
    - Stemming
    - and... anything else you think it's needed (bonus point)

In [ ]:
import json
import re
import sys

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

for pkg in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

STOP_WORDS = set(stopwords.words("english"))
STEMMER = PorterStemmer()

# Fields the final search output must return (task 2)
OUTPUT_FIELDS = ["pid", "title", "description", "brand", "category", "sub_category",
                 "product_details", "seller", "out_of_stock", "selling_price",
                 "discount", "actual_price", "average_rating", "url"]


# ---------------------------------------------------------------- text
def build_terms(text):
    """lowercase -> strip urls -> remove punctuation -> tokenize -> stopwords -> stem."""
    if not isinstance(text, str) or not text.strip():
        return []
    text = text.lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # punctuation -> space ("look.Designed" -> 2 tokens)
    tokens = text.split()                      # after cleaning, whitespace split == tokenization
    tokens = [t for t in tokens if t not in STOP_WORDS]
    tokens = [t for t in tokens if len(t) > 1 or t.isdigit()]  # drop stray single letters
    return [STEMMER.stem(t) for t in tokens]


# ------------------------------------------------------------- numeric
def to_float(value):
    """'2,999' -> 2999.0 ; '69% off' -> 69.0 ; '' / None -> None."""
    if value is None:
        return None
    m = re.search(r"\d[\d,]*\.?\d*", str(value))
    if not m:
        return None
    return float(m.group().replace(",", ""))


# ------------------------------------------------------ product_details
def flatten_details(details):
    """[{'Fabric': 'Cotton'}, {'Color': 'Blue'}] -> {'Fabric': 'Cotton', 'Color': 'Blue'}"""
    flat = {}
    for item in details or []:
        if isinstance(item, dict):
            flat.update({str(k): str(v) for k, v in item.items()})
    return flat


def details_text(flat):
    # Skip identifier-like keys that add noise, not meaning
    skip = {"style code", "sku", "model number", "pack of"}
    parts = [f"{k} {v}" for k, v in flat.items() if k.lower() not in skip]
    return " ".join(parts)


# --------------------------------------------------------------- docs
def load_docs(path):
    with open(path, encoding="utf-8") as f:
        first = f.read(1)
        f.seek(0)
        if first == "[":                      # JSON array
            return json.load(f)
        return [json.loads(line) for line in f if line.strip()]  # JSON lines


def process_doc(doc):
    flat = flatten_details(doc.get("product_details"))
    processed = {
        # --- original info, kept for displaying results (pid kept for evaluation)
        "pid": doc.get("pid"),
        "title": doc.get("title"),
        "description": doc.get("description"),
        "brand": doc.get("brand"),
        "category": doc.get("category"),
        "sub_category": doc.get("sub_category"),
        "product_details": doc.get("product_details"),
        "seller": doc.get("seller"),
        "out_of_stock": doc.get("out_of_stock"),
        "selling_price": to_float(doc.get("selling_price")),
        "discount": to_float(doc.get("discount")),
        "actual_price": to_float(doc.get("actual_price")),
        "average_rating": to_float(doc.get("average_rating")),
        "url": doc.get("url"),
        # --- searchable fields (separate fields, same analyzer)
        "title_terms": build_terms(doc.get("title")),
        "description_terms": build_terms(doc.get("description")),
        "brand_terms": build_terms(doc.get("brand")),
        "category_terms": build_terms(f"{doc.get('category') or ''} {doc.get('sub_category') or ''}"),
        "details_terms": build_terms(details_text(flat)),
        "seller_terms": build_terms(doc.get("seller")),
    }
    # Combined bag of words (useful as a baseline / for TF-IDF over everything)
    processed["all_terms"] = (processed["title_terms"] + processed["description_terms"]
                              + processed["brand_terms"] + processed["category_terms"]
                              + processed["details_terms"])
    return processed


def main(src, dst):
    docs = load_docs(src)
    seen, out = set(), []
    for d in docs:
        pid = d.get("pid")
        if pid in seen:          # drop duplicate products
            continue
        seen.add(pid)
        out.append(process_doc(d))
    with open(dst, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    print(f"{len(docs)} raw docs -> {len(out)} processed docs saved to {dst}")


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "fashion_products_dataset.json",
         sys.argv[2] if len(sys.argv) > 2 else "processed_docs.json")